# Appendix 05: Least-squares Minimization

Source orientation: printed pages 588-596; PDF pages 606-614.

This notebook is an original, standalone computational treatment of the appendix. The source pages were used only to identify the section hierarchy and mathematical targets: linear least squares, normal equations, total least squares, weighted least squares, constrained systems, and sparse symmetric equations.

## Standalone Learning Goals

By the end of this appendix lab you should be able to:

- read an overdetermined linear system as a geometric projection onto a column space;
- explain why the normal equations solve the same problem but square the condition number;
- distinguish ordinary least squares from total least squares in a line-fitting problem;
- use weights to encode unequal measurement certainty;
- recognize sparse symmetric normal systems as graph-structured geometry rather than anonymous arrays.

## Library Routing

The appendix is mostly linear algebra, so NumPy and SciPy are the right computational core. Matplotlib is used for durable inspection figures: projection geometry, conditioning, line-fit residuals, and sparse normal structure. SciPy sparse exposes the matrix pattern directly, while NetworkX turns the same sparsity into an incidence graph so the learner can see why large reconstruction systems are solved blockwise rather than as dense equations.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

artifact_paths = []


## Concept Map From Equations To Geometry

A least-squares problem starts with equations that cannot all be true at once:

$$Ax \approx b, \qquad \min_x \|Ax-b\|_2^2.$$

The solution is the point in the column space of `A` closest to `b`. The residual `r = b - A x_hat` is not random leftover error: it is orthogonal to every column of `A`, so `A.T @ r = 0`. Normal equations, SVD solves, weighted least squares, total least squares, and sparse solvers are different ways of preserving or modifying this geometric projection.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from scipy import sparse
from scipy.linalg import null_space

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib

TOPIC = "appendix-05"
rng = np.random.default_rng(105)

plt.rcParams.update({"figure.dpi": 120})


## 1. Projection: the residual is a perpendicular witness

The first figure uses a two-dimensional column space so the geometry is inspectable. The blue line is all possible predictions `Ax`; the observed vector `b` is off the line; the least-squares prediction is the foot of the perpendicular. The learner should inspect that the residual arrow lands at right angles to the column-space direction, which is exactly the normal-equation condition.

In [ ]:
a = np.array([[1.0], [0.48]])
b = np.array([2.4, 2.35])
x_hat = np.linalg.lstsq(a, b, rcond=None)[0]
projection = (a @ x_hat).ravel()
residual = b - projection
orthogonality = float(abs((a.T @ residual)[0]))

fig, ax = plt.subplots(figsize=(7, 6))
t = np.linspace(-0.2, 3.2, 80)
line = t[:, None] * a.ravel()
ax.plot(line[:, 0], line[:, 1], color="#2f6fbb", lw=2.5, label="column space of A")
ax.scatter([0, b[0], projection[0]], [0, b[1], projection[1]], color=["#444", "#b23a48", "#1f7a4d"], zorder=5)
ax.annotate("observed b", b + [0.05, 0.04], color="#b23a48")
ax.annotate("least-squares Ax", projection + [0.04, -0.14], color="#1f7a4d")
ax.arrow(0, 0, b[0], b[1], head_width=0.06, length_includes_head=True, color="#b23a48", alpha=0.55)
ax.arrow(0, 0, projection[0], projection[1], head_width=0.06, length_includes_head=True, color="#1f7a4d", alpha=0.7)
ax.arrow(projection[0], projection[1], residual[0], residual[1], head_width=0.06, length_includes_head=True, color="#d28b26", alpha=0.9, label="residual r")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)
ax.set_title("Least-squares projection and orthogonal residual")
ax.legend(loc="upper left")
projection_path = save_matplotlib(fig, TOPIC, "figures", "least-squares-projection.png")
plt.close(fig)
artifact_paths.append(projection_path)
display_artifact(projection_path, width=760)
print({"A.T_r_abs": orthogonality})


## 2. Normal equations: same answer, worse conditioning

The normal equations `A.T A x = A.T b` are often attractive because the system is square and symmetric. Their hidden cost is visible in the singular values: `cond(A.T A)` is approximately `cond(A)^2`. For well-conditioned data this is harmless; for nearly dependent columns it can turn a stable geometric fit into a fragile numerical problem.

In [ ]:
condition_targets = np.array([1e1, 1e2, 1e3, 1e4])
conditioning_rows = []
for kappa in condition_targets:
    U, _ = np.linalg.qr(rng.normal(size=(40, 2)))
    V, _ = np.linalg.qr(rng.normal(size=(2, 2)))
    A = U @ np.diag([1.0, 1.0 / kappa]) @ V.T
    b2 = rng.normal(size=40)
    x_svd = np.linalg.lstsq(A, b2, rcond=None)[0]
    x_ne = np.linalg.solve(A.T @ A, A.T @ b2)
    conditioning_rows.append({
        "cond_A": float(np.linalg.cond(A)),
        "cond_ATA": float(np.linalg.cond(A.T @ A)),
        "solution_gap": float(np.linalg.norm(x_svd - x_ne)),
    })

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.loglog([r["cond_A"] for r in conditioning_rows], [r["cond_ATA"] for r in conditioning_rows], "o-", label="observed cond(A.T A)")
ax.loglog(condition_targets, condition_targets ** 2, "--", label="cond(A)^2 reference")
for row in conditioning_rows:
    ax.annotate(f"gap {row['solution_gap']:.1e}", (row["cond_A"], row["cond_ATA"]), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("condition number of A")
ax.set_ylabel("condition number of A.T A")
ax.set_title("Normal equations square the conditioning problem")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
conditioning_path = save_matplotlib(fig, TOPIC, "figures", "normal-equations-conditioning.png")
plt.close(fig)
artifact_paths.append(conditioning_path)
display_artifact(conditioning_path, width=760)


## 3. Ordinary LS versus TLS: vertical error is not geometric error

Ordinary least squares treats the horizontal coordinate as exact and measures vertical residuals. Total least squares treats both coordinates as noisy and chooses the line whose orthogonal distances are smallest. In image geometry this distinction matters because measurements usually live in the image plane, not in a privileged dependent variable.

In [ ]:
true_m, true_c = 0.72, -0.4
x = np.linspace(-2.4, 2.4, 34)
y = true_m * x + true_c + rng.normal(scale=0.22, size=x.size)
x_noisy = x + rng.normal(scale=0.18, size=x.size)

ols_m, ols_c = np.polyfit(x_noisy, y, 1)
points = np.column_stack([x_noisy, y])
center = points.mean(axis=0)
_, _, vh = np.linalg.svd(points - center)
tls_direction = vh[0]
tls_normal = vh[1]
tls_m = -tls_normal[0] / tls_normal[1]
tls_c = center[1] - tls_m * center[0]

xx = np.linspace(x_noisy.min() - 0.3, x_noisy.max() + 0.3, 100)
fig, ax = plt.subplots(figsize=(7.5, 5.2))
ax.scatter(x_noisy, y, s=28, color="#30343f", label="noisy 2D measurements")
ax.plot(xx, ols_m * xx + ols_c, color="#2f6fbb", lw=2.5, label="ordinary LS")
ax.plot(xx, tls_m * xx + tls_c, color="#b23a48", lw=2.5, label="total LS")
for idx in [5, 13, 21, 29]:
    ax.plot([x_noisy[idx], x_noisy[idx]], [y[idx], ols_m * x_noisy[idx] + ols_c], color="#2f6fbb", alpha=0.45)
    signed = (points[idx] - center) @ tls_normal
    foot = points[idx] - signed * tls_normal
    ax.plot([points[idx, 0], foot[0]], [points[idx, 1], foot[1]], color="#b23a48", alpha=0.55)
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)
ax.set_title("TLS measures orthogonal distance; LS measures vertical distance")
ax.legend()
tls_path = save_matplotlib(fig, TOPIC, "figures", "total-least-squares-line-fit.png")
plt.close(fig)
artifact_paths.append(tls_path)
display_artifact(tls_path, width=760)

tls_smallest_sv = float(np.linalg.svd(points - center, compute_uv=False)[-1])
tls_normal_residual = float(np.max(np.abs((points - center) @ tls_normal)))


## 4. Weights and sparsity: residual geometry has structure

Weights say which residual directions are trusted. Sparse symmetric normal systems say which unknowns interact. Both are central in multiple-view estimation: a high-confidence measurement should pull harder, and a camera-point problem should expose its bipartite structure instead of becoming a featureless dense matrix.

In [ ]:
xw = np.linspace(-1.0, 1.0, 18)
Aw = np.column_stack([xw, np.ones_like(xw)])
true = np.array([1.4, -0.25])
yw = Aw @ true + rng.normal(scale=0.04, size=xw.size)
yw[-4:] += np.array([0.55, -0.45, 0.65, -0.5])
weights = np.ones_like(yw)
weights[-4:] = 0.12
x_unweighted = np.linalg.lstsq(Aw, yw, rcond=None)[0]
Wsqrt = np.sqrt(weights)[:, None]
x_weighted = np.linalg.lstsq(Wsqrt * Aw, Wsqrt.ravel() * yw, rcond=None)[0]
high_confidence_residual_unweighted = float(np.linalg.norm((Aw @ x_unweighted - yw)[:-4]))
high_confidence_residual_weighted = float(np.linalg.norm((Aw @ x_weighted - yw)[:-4]))

# A tiny camera-point style Jacobian pattern: two cameras, four points, each observation touches one camera and one point.
sparse_rows = []
sparse_cols = []
for obs, (cam, pt) in enumerate((c, p) for c in range(2) for p in range(4)):
    base = 2 * obs
    for local in range(2):
        sparse_rows.append(base + local); sparse_cols.append(2 * cam + local)
        sparse_rows.append(base + local); sparse_cols.append(4 + 2 * pt + local)
J = sparse.coo_matrix((np.ones(len(sparse_rows)), (sparse_rows, sparse_cols)), shape=(16, 12)).tocsr()
N = (J.T @ J).tocsr()
sparse_symmetry = float((N - N.T).nnz)

G = nx.Graph()
for c in range(2): G.add_node(f"camera {c+1}", bipartite=0)
for p in range(4): G.add_node(f"point {p+1}", bipartite=1)
for c in range(2):
    for p in range(4):
        G.add_edge(f"camera {c+1}", f"point {p+1}")
pos = {f"camera {c+1}": (0, 1-c) for c in range(2)} | {f"point {p+1}": (2, 1.5-p) for p in range(4)}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
axes[0].scatter(xw[:-4], yw[:-4], color="#30343f", label="trusted")
axes[0].scatter(xw[-4:], yw[-4:], color="#d28b26", label="downweighted")
axes[0].plot(xw, Aw @ x_unweighted, color="#2f6fbb", label="unweighted")
axes[0].plot(xw, Aw @ x_weighted, color="#b23a48", label="weighted")
axes[0].set_title("weights move the fit toward trusted residuals")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.25)
axes[1].spy(N, markersize=7, color="#2f6fbb")
axes[1].set_title("sparse symmetric normal matrix")
axes[2].set_title("same pattern as an observation graph")
nx.draw_networkx(G, pos=pos, ax=axes[2], node_size=850, font_size=8, edge_color="#888", node_color=["#8ecae6" if n.startswith("camera") else "#ffb703" for n in G.nodes])
axes[2].axis("off")
fig.tight_layout()
sparse_path = save_matplotlib(fig, TOPIC, "figures", "weighted-sparse-normal-system.png")
plt.close(fig)
artifact_paths.append(sparse_path)
display_artifact(sparse_path, width=960)


## Worked Example Reading Guide

Read the four examples as one chain rather than four isolated plots. The projection diagram gives the geometric certificate for ordinary least squares: the residual must be perpendicular to the column space, so a residual vector that still has a component along any column of `A` cannot be optimal. The conditioning panel explains why a mathematically equivalent normal-equation solve can be a worse computational choice; when the columns of `A` are nearly dependent, the squared condition number makes small measurement perturbations look much larger in the solution.

The line-fitting panel then asks what kind of error the model is measuring. Ordinary least squares is appropriate when the horizontal coordinate is trusted and the vertical coordinate is noisy. Total least squares is more faithful when both image coordinates are measured quantities, because the shortest distance to a line is orthogonal, not vertical. That distinction is easy to miss in algebra but obvious in the residual segments.

The final sparse-system view connects the appendix back to multiple-view geometry. Weights encode confidence in residuals, while sparsity records which cameras, points, or parameters share an observation. A dense solver can still produce a number, but it hides the reason bundle-adjustment systems are tractable: most variables never meet directly. The acceptance target for this appendix is therefore not memorizing one solver. It is being able to say which geometry the residual measures, what uncertainty the weights encode, and what graph the normal matrix represents.

## Applied Lab

Replace the synthetic line measurements with residuals from a small geometric estimator, such as point-to-line distances after a homography or epipolar-line fit. Keep three views of the computation: the projection residual, the conditioning diagnostic, and the weighted residual comparison. The key lab question is whether the residual you minimize is the geometric residual you actually care about.

In [ ]:
invariants = {
    "topic": TOPIC,
    "source_span": {"printed": "588-596", "pdf": "606-614"},
    "library_route": ["numpy.linalg", "scipy.sparse", "networkx", "matplotlib"],
    "artifacts": [str(path.relative_to(BOOK_ROOT)).replace("\\", "/") for path in artifact_paths],
    "orthogonality_Atr_abs": orthogonality,
    "condition_square_ratio": float(conditioning_rows[-1]["cond_ATA"] / (conditioning_rows[-1]["cond_A"] ** 2)),
    "tls_smallest_singular_value": tls_smallest_sv,
    "max_tls_orthogonal_residual": tls_normal_residual,
    "weighted_high_confidence_residual": high_confidence_residual_weighted,
    "unweighted_high_confidence_residual": high_confidence_residual_unweighted,
    "sparse_normal_nnz": int(N.nnz),
    "sparse_normal_symmetry_nnz": int(sparse_symmetry),
}
summary_path = save_json(invariants, TOPIC, "checks", "least-squares-invariants.json")
artifact_paths.append(summary_path)
display_artifact(summary_path)
assert_artifacts(artifact_paths, min_bytes=64)
assert invariants["orthogonality_Atr_abs"] < 1e-12
assert 0.9 < invariants["condition_square_ratio"] < 1.1
assert high_confidence_residual_weighted < high_confidence_residual_unweighted
assert invariants["sparse_normal_symmetry_nnz"] == 0
final_sanity = invariants
invariants


## How To Use These Checks In A Chapter Notebook

When a chapter estimates a homography, a camera matrix, or a fundamental matrix, this appendix gives the checklist for deciding whether the linear solve is trustworthy. First ask what the columns of the design matrix mean geometrically. If the columns encode nearly the same constraint, the condition plot predicts unstable parameters even when the algebraic residual is small. Next ask whether the residual is measured in the right space. A coefficient-space residual may be convenient for a nullspace solve, but an image-space residual is usually what a learner can inspect and what an algorithm should ultimately reduce. Finally ask what noise model is being assumed. Equal weights claim equal uncertainty; unequal weights claim that some measurements are more reliable; total least squares claims uncertainty in the design data as well as the observations. These are modeling decisions, not cosmetic solver options.

## Inspection Questions

Before accepting a least-squares answer, ask three questions. Is the residual orthogonal to the intended model space? Does the condition number warn that the parameters are sensitive to tiny perturbations? Are the plotted residuals measured in the same coordinates as the geometric claim? If any answer is unclear, the numerical solution needs a better diagnostic before it should be used in a chapter lab.

## Pitfalls And Misconceptions

- A small algebraic residual is not automatically the same as a small geometric residual.
- The normal equations are not a harmless rewrite when `A` is ill-conditioned.
- Total least squares is not just ordinary least squares with a different plotting style; it changes which measurement directions are treated as uncertain.
- Weights must describe measurement uncertainty, not the answer the estimator is supposed to produce.
- Sparse systems are not merely faster dense systems; their sparsity records which variables actually interact.

## Takeaways

- Least squares is a projection theorem made executable.
- The residual is meaningful because it is orthogonal to the model space.
- SVD is the safer default for fragile linear systems.
- TLS, weighted LS, and sparse normal systems are different geometric statements about noise and coupling.
- MVG estimation improves when every residual is tied to the measurement geometry it claims to model.